# 02. How is text converted to numbers and vectors?

In lesson 00, you got acquainted with tensor, vector and learning a number. In lesson 01, you saw that the language model calculates the probability of the next token to continue the text. This lesson answers two questions before that calculation: "What exactly is a token?" and "Why is the token id converted to a vector?"

**Token** is one of the pieces into which the text is divided. This fragment can be a character, part of a word, a word or multiple bytes. The splitting method is determined by the Tokenizer. There is no general rule that every token is equal to a word.

**Study prerequisite:** [lesson 00] (00_start_here.ipynb) and [lesson 01] (01_language_model_basics.ipynb). This file is independent in terms of execution; The variables of those two files do not need to be in memory. Run the code cells from top to bottom. If you restart the kernel, start the lesson from the beginning.

First, you check the Python string itself. Then you create a character tokenizer, run the frequent segment merge method, and end up with a table of trainable vectors. You can read the couple integration section in a separate session.

## 1. Prepare the execution environment

The notebook kernel may be run from the `notebooks` folder or from the project root. To read the common code of the lessons, you first find the folder where the path `src/tinyllm` is located.

The `Path` object represents a file or folder path. The `parents` section provides higher folders. The `for` loop checks them one by one and the `break` command ends the loop when the project is found. This is the search of parents; Username or installation path is not fixed in the code.

Import the PyTorch library with the name `torch`. The command to determine the random seed or `seed` helps to repeat the random numbers of the examples in the same environment. The exact same result is not guaranteed on every hardware.

In [ ]:
from pathlib import Path  # The paths tool is from the Python standard library.
import sys               # Provides search paths for Python modules.
import torch             # It is a tool for building tensors and numerical calculations.

# First, we check the current folder and then the higher folders.
ROOT = None
candidates = [Path.cwd()] + list(Path.cwd().parents)
for candidate in candidates:
    if (candidate / "src" / "tinyllm").is_dir():
        ROOT = candidate
        break

if ROOT is None:
    raise RuntimeError("Run this notebook from inside the tiny-llm-fa folder.")

# We add the path of the project codes to the import search.
source_path = str(ROOT / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)

torch.set_num_threads(2)  # For small calculations, two execution threads are enough.
torch.manual_seed(7)     # We fix the starting point of generating random numbers.
print("Project:", ROOT.name)
print("PyTorch:", torch.__version__)

The project name in the output should be `tiny-llm-fa`. If you see a folder not found error, open the notebook file from within the same project. This cell has not trained any model.

## 2. Before tokenization: string, character and byte

In Python, text like "hello" is a string. Looping over a string returns units called Code points. The **Unicode** standard defines specific numbers for these units. For example, "Y" in Persian and its similar letter in Arabic have two different numbers.

The number of these units is not always equal to the number of signs you see with your eyes. A letter with a symbol may be written with a code point or by combining a letter and a symbol. A semispace is also an independent unit, although it does not create a large empty space on the screen.

**Objective of the example:** See the difference between the two strings "I go" and "I go" in memory. The function `repr` returns the checkable representation of the string; It shows some invisible characters in the form of code. The function `ord` returns the Unicode number of a character, and `hex` represents that number in hexadecimal.

In [ ]:
import unicodedata  # Provides name information and Unicode rules.

with_zwnj = "می‌روم"     # There is a half space between May and Rome.
without_zwnj = "میروم"  # This field does not have a half-space.
print("With ZWNJ:", repr(with_zwnj), "length:", len(with_zwnj))
print("Without ZWNJ:", repr(without_zwnj), "length:", len(without_zwnj))

for character in with_zwnj:
    # The official Unicode name helps identify the invisible character.
    print(repr(character), hex(ord(character)), unicodedata.name(character))

The length of the first string is 6 and the length of the second string is 5. In the list of the first string, you will see the name `ZERO WIDTH NON-JOINER`; This is the half distance. For this course's character tokenizer, a semispace will also be a token if it appears in the training data.

**Byte** is the unit of data storage in the file and data transfer; The value of a byte is from 0 to 255. The **UTF-8** method converts Unicode text to bytes. A Persian character usually needs a few bytes. Therefore, the length of the string and the number of bytes stored in it are two different concepts.

The `encode` method on the string constructs the bytes. The `decode` method on bytes reconstructs the text. These two names are also used in the token generator, but their input and output are different: there you will work with the token ID.

In [ ]:
for text in ["A", "س", "سلام", "😀"]:
    encoded_bytes = text.encode("utf-8")  # We convert the string to bytes.
    restored_text = encoded_bytes.decode("utf-8")  # We return the bytes to the string.
    print(repr(text), "code points:", len(text), "bytes:", list(encoded_bytes))
    print("Byte count:", len(encoded_bytes), "restored:", restored_text)
    assert restored_text == text  # This check returns an error if the reconstruction is not accurate.

For "Hello", the number of code points is 4 and the number of bytes is 8. The emoji in this example has one code point and four bytes. Do not generalize this result to all emojis; Some visual symbols are made up of multiple code points.

## 3. Normalization: what to normalize?

**Normalization** means converting multiple input shapes into one selected shape. For example, if you match two different representations of a signed letter, the program will not treat them as two separate entries.

One of the standard Unicode rules is the form `NFC`. This form can convert a combination of a letter and a symbol into a combined representation. But replacing the Arabic letter with Persian is a separate rule; The form `NFC` does not do that.

**Example Objective:** Find both the seemingly invisible difference and see the limit of `NFC`'s ability.

In [ ]:
accent_one = "é"        # This display has a code point.
accent_two = "e\u0301"  # The letter e and the separate sign have two code points.
print("Before NFC:", accent_one == accent_two)
print("Lengths:", len(accent_one), len(accent_two))

normalized = unicodedata.normalize("NFC", accent_two)
print("After NFC:", normalized == accent_one)

persian_y = "ی"
arabic_y = "ي"  # This letter is intentionally Arabic to test the difference.
print("Y code points:", hex(ord(persian_y)), hex(ord(arabic_y)))
print("NFC replaces Arabic Y:", unicodedata.normalize("NFC", arabic_y) == persian_y)
print("K code points:", hex(ord("ک")), hex(ord("ك")))

The first check is `False` and the check after `NFC` is equal to `True`. The analysis of the conversion of "y" from Arabic to Farsi remains `False`. Similarity on the screen does not mean equality of strings.

In this project, hidden normalization is not done; You tokenize the text as received. If you add a rule later, such as converting Arabic letters or removing extra space, apply the same rule before training and when the model answers. Otherwise, the model receives two different shapes from the input.

## 4. Build the character tokenizer first with a dictionary

**Vocabulary** is a collection of tokens that the token maker knows. In the character method, each known character is a member of this set. **ID** is just a number that identifies that member. For example, you can consider "A" as ID 1 and "B" as ID 2. The size of this number does not indicate the size or importance of the letter.

A Python dictionary holds the value of a key. In the `stoi` dictionary, the key is a character and the value is its identifier. This short name comes from the phrase "string to integer".

**Example Goal:** Convert the text "Bob" to a list of IDs and then back to text.

In [ ]:
small_chars = ["ا", "ب", "م"]  # The small vocabulary of this example has only three characters.
small_stoi = {}                 # An empty dictionary for character-to-id mapping.
for index, character in enumerate(small_chars):
    # The enumerate function gives both the position number and the value of that position.
    small_stoi[character] = index + 1  # We leave the zero identifier for the unknown.

text = "باب"
small_ids = []
for character in text:
    small_ids.append(small_stoi[character])  # We add the ID to the end of the list.

restored_characters = []
for token_id in small_ids:
    restored_characters.append(small_chars[token_id - 1])
restored = "".join(restored_characters)  # We paste the characters together without a separator.
print("Vocabulary:", small_stoi)
print("IDs:", small_ids)
print("Restored:", restored)
assert restored == text

IDs should be `[2, 1, 2]`. Because Python lists start at zero and the IDs in this example start at one, you subtracted one when you returned. **ID 2 appears twice; But you only need one row for "b" in the table.**

If you switch the identifiers "a" and "b", the tokenizer is still constructable. But a model trained with a previous mapping interprets these identifiers in the same way as before. For this reason, the tokenizer is part of the essential files of a stored model.

## 5. From the small example to the real project class

A class holds together some related data and operations. The `CharTokenizer` class holds a list of characters and has three main operations: building a vocabulary from text with `fit`, converting text to identifiers with `encode`, and reconstructing text with `decode`.

In the following code, you will see the same behavior of the project class with more comments. The name `ExplainedCharTokenizer` was chosen only to avoid interference with the original class. This code is not just text display; You run it and create an object in the cell after it.

The `__init__` method is executed when the object is created. The name `self` refers to the same object. The `@classmethod` symbol says that the `fit` method has access to the class itself; The name `cls` is the representative of the class. So calling `cls(...)` creates a new tokenizer.

You know the unknown token with the name `unk`; Abbreviation is `unknown`. ID zero is reserved for it. Strict mode or `strict=True` gives an error instead of hiding the unknown character.

In [ ]:
class ExplainedCharTokenizer:
    def __init__(self, chars):
        self.chars = list(chars)  # The order of the characters determines the meaning of the identifiers.
        # set removes duplicates; The difference in length means that the entry was repeated.
        if len(set(self.chars)) != len(self.chars):
            raise ValueError("Characters must be unique.")
        for character in self.chars:
            if len(character) != 1:
                raise ValueError("Each entry must be one Unicode code point.")
        self.stoi = {}
        for index, character in enumerate(self.chars):
            self.stoi[character] = index + 1
        self.vocab_size = len(self.chars) + 1  # We have an additional member for unknown ID.

    @classmethod
    def fit(cls, text):
        unique_chars = set(text)  # We remove the repetition of each letter in the text.
        ordered_chars = sorted(unique_chars)  # We make a specific and repeatable order.
        return cls(ordered_chars)

    def encode(self, text, strict=True):
        unknown = sorted(set(text) - set(self.chars))  # The difference is two sets.
        if unknown and strict:
            raise ValueError(f"Characters outside the trained vocabulary: {unknown!r}")
        ids = []
        for character in text:
            ids.append(self.stoi.get(character, 0))  # get returns zero if there is no key.
        return ids

    def decode(self, ids):
        characters = []
        for token_id in ids:
            if int(token_id) == 0:
                characters.append("�")  # alternative sign; The original letter can no longer be recovered.
            else:
                characters.append(self.chars[int(token_id) - 1])
        return "".join(characters)

    def to_dict(self):
        # This information is enough to save and restore the map.
        return {"type": "character", "unk_id": 0, "chars": self.chars}

The class is defined, but no vocabulary is created until you create an object from it. The `fit` method in this class only collects text characters; This operation is different from training the weights of a neural network. The name `fit` in machine learning tools usually refers to building or tuning a component from data.

**Example Objective:** See the known and unknown character behavior separately. The `try/except` section will catch the error that you intentionally cause so that the rest of the lesson doesn't stop running.

In [ ]:
mini_tokenizer = ExplainedCharTokenizer.fit("باب مام")
print("Vocabulary:", mini_tokenizer.to_dict())
print("Known text:", mini_tokenizer.encode("باب"))
print("Round trip:", mini_tokenizer.decode(mini_tokenizer.encode("باب")))

try:
    mini_tokenizer.encode("باب؟")  # The question mark was not in the vocabulary building text.
except ValueError as error:
    print("Expected error:", error)

unknown_ids = mini_tokenizer.encode("باب؟", strict=False)
print("Non-strict IDs:", unknown_ids)
print("Non-strict restored:", mini_tokenizer.decode(unknown_ids))
assert unknown_ids[-1] == 0

In non-strict mode, the last identifier is zero and the reconstructed text ends with the token `�`. Round trip is valid only when all characters are known. The unknown identifier does not hold the original character information.

### From which data should we make vocabulary?

**Training** data is used to build learnable components. **Validation** data is for checking choices and settings. **Test** data is set aside for final evaluation. You will see the details of separation in lesson 05.

In this course, the vocabulary is made only from the teaching text. The new character in the other two sections can be unknown; You should see its number. Vocabulary construction from all sections has incorporated information from the evaluation data into the construction phase and changed the testing boundary.

**Code Objective:** Read the actual tokenizer and actual course data, reconstruct a text, and compare the behavior of the described version with the original class. The function `load_data` returns four values; Assigning four names keeps each value separate.

In [ ]:
from tinyllm.data import CharTokenizer, load_data

# Texts are read from the local file; This course does not have internet download.
documents, split_texts, tokenizer, token_tensors = load_data(ROOT / "data")
explained = ExplainedCharTokenizer.fit(split_texts["train"])
assert explained.to_dict() == tokenizer.to_dict()  # The mapping of the two implementations is identical.

sample_text = "model"
sample_ids = tokenizer.encode(sample_text)
print("Vocabulary size:", tokenizer.vocab_size)
print("Sample IDs:", sample_ids)
print("Restored:", tokenizer.decode(sample_ids))
assert tokenizer.decode(sample_ids) == sample_text

for split_name, ids in token_tensors.items():
    unknown_count = int((ids == 0).sum().item())  # The count of identifiers is unknown.
    print(split_name, "tokens:", ids.numel(), "unknown:", unknown_count)

Vocabulary size is the number of distinct training characters plus an unknown member. You can check the "model" identifiers with the real dictionary; These identifiers are not the same as the numbers in the "Bob" example, because the vocabulary is different.

The main class is in the file `src/tinyllm/data.py`. The original version abbreviated some loops with **List comprehension**; For example, the expression `[self.stoi.get(ch, 0) for ch in text]` is the same list construction as you did in the `encode` loop. Being shorter does not change the algorithm.

## 6. Why is the character not the only possible choice?

The character token generator is simple and you can see all its steps. But long text creates a longer list of identifiers. Part-of-word tokenization can include multiple characters in a single token. In this case, the number of input positions decreases and the vocabulary becomes larger.

The **BPE** method, abbreviated as `Byte Pair Encoding`, starts with small units and merges frequent adjacent pairs. Each merge creates a new ID. In the next example, the work base is bytes; All 256 byte values ​​have identifiers from the beginning. This option helps to break new text into bytes without unknown tokens.

This is a separate token generator for learning. The small period model still uses `CharTokenizer`. Do not pass the identifiers of this BPE directly to that model; The meaning of the identifiers and the vocabulary size of the two systems are different.

### Step 1: Replace a pair from left to right

Suppose the input IDs are `[1, 1, 1]` and merge the pair `(1, 1)` with ID 9. The result should be `[9, 1]`. The first two members are consumed; It is not possible to use the second member in another pair at the same time.

In the next function, the variable `position` holds the current position. The `while` loop repeats until the condition is true. After merging, you move forward two places, otherwise one place. The name `pair` is a pair or **Tuple**; It means two values ​​together like `(1, 1)`.

In [ ]:
def merge_pair(sequence, pair, new_id):
    result = []
    position = 0
    while position < len(sequence):
        # First we check if the next member exists so we don't cross the end of the list.
        has_next = position + 1 < len(sequence)
        if has_next and (sequence[position], sequence[position + 1]) == pair:
            result.append(new_id)  # We replace two members with a new identifier.
            position += 2          # We discard both consumed members.
        else:
            result.append(sequence[position])  # This member remains unchanged.
            position += 1
    return result

print("One merge:", merge_pair([1, 1, 1], (1, 1), 9))
print("Two merges:", merge_pair([1, 1, 1, 1], (1, 1), 9))
assert merge_pair([1, 1, 1], (1, 1), 9) == [9, 1]

The first result is `[9, 1]` and the second result is `[9, 9]`. The function only replaces the pair you give it; Choosing the most frequent pair is the next step.

### Step 2: Count adjacent pairs

The `Counter` tool is a counter dictionary. When you increment a key, the occurrence count of that key is kept. `(a, b)` pair and `(b, a)` pair are two different keys; Order is important.

The following function checks each text separately. The end of a text is not connected to the beginning of the next text; Therefore, a pair will not be created from the border between two texts.

In [ ]:
from collections import Counter

def count_pairs(sequences):
    counts = Counter()
    for sequence in sequences:
        # For a list of length three, we have two adjacent pairs: positions zero and one.
        for position in range(len(sequence) - 1):
            pair = (sequence[position], sequence[position + 1])
            counts[pair] += 1
    return counts

print("Pair counts:", count_pairs([[1, 1, 1], [1, 2]]))
assert count_pairs([[1, 1, 1]])[(1, 1)] == 2

The count of the `(1, 1)` pair in the three-member list is 2, but the number of non-overlapping merges in the same list was only 1. Counting candidates and replacing them are two different stages.

### Step 3: Learn the integration rules from several texts

The `pieces` dictionary specifies which bytes each identifier points to. The `merges` index holds the merge rules in learning order. IDs 0 to 255 are for base bytes; Merge IDs start at 256.

In selecting the highest count, if two pairs are equal, the pair with the higher numerical order is selected. This is just a decision to make the example consistent. The expression `lambda pair: ...` is a short function that returns the comparison key; The first value is the number and the second value is even.

**Code goal:** Make up to 16 rules and see the content of each piece. The function returns separate outputs so you can later change the number of integrations without manipulating global variables.

In [ ]:
def train_byte_bpe(training_texts, max_merges):
    sequences = []
    for text in training_texts:
        sequences.append(list(text.encode("utf-8")))  # Each text becomes a list of bytes.

    pieces = {}
    for byte_id in range(256):
        pieces[byte_id] = bytes([byte_id])  # The base segment is exactly one byte.
    merges = []

    for new_id in range(256, 256 + max_merges):
        counts = count_pairs(sequences)
        if not counts or max(counts.values()) < 2:
            break  # If there are no frequent pairs left, we finish training the rules.
        pair = max(counts, key=lambda candidate: (counts[candidate], candidate))
        pieces[new_id] = pieces[pair[0]] + pieces[pair[1]]  # The bytes of the two pieces are concatenated.
        merges.append((pair, new_id))  # The order of these rules is important for later use.

        updated_sequences = []
        for sequence in sequences:
            updated_sequences.append(merge_pair(sequence, pair, new_id))
        sequences = updated_sequences
    return pieces, merges

training_texts = ["low lower lowest low", "newer wider lower", "hello world hello friend"]
bpe_pieces, bpe_merges = train_byte_bpe(training_texts, max_merges=16)
for pair, new_id in bpe_merges:
    # The representation of bytes is valid, even if a fragment is not complete UTF-8 text by itself.
    print("ID:", new_id, "pair:", pair, "bytes:", repr(bpe_pieces[new_id]))
print("Vocabulary size:", len(bpe_pieces))

Each line is a rule: if those two identifiers are next to each other, they are replaced by the new identifier. The new fragment can consist of several bytes and may not be a complete word. Seeing Persian bytes in the form of `\x...` in the output is not a text error; This is the Python representation for bytes.

This tutorial implementation converts all text directly to bytes. Real tools usually have a pre-tokenization stage; For example, the text is first divided into parts with spacing and punctuation rules. They also have **Special token** for signs such as the end of the text or message border. None are added in this example, and this code does not have a dialog template.

### Step 4: Apply the fixed rules to the new text

You don't relearn the rules from input when using. First, the new text is converted to bytes, and then the stored rules are applied in the same order. For reconstruction, all byte chunks are concatenated and read only once with UTF-8.

If you convert each piece to a string separately, it might be half a multibyte character and you'll get an error. The token identifier does not necessarily respect the boundary of a Unicode character.

In [ ]:
def bpe_encode(text, merges):
    sequence = list(text.encode("utf-8"))
    for pair, new_id in merges:
        sequence = merge_pair(sequence, pair, new_id)
    return sequence

def bpe_decode(ids, pieces):
    chunks = []
    for token_id in ids:
        chunks.append(pieces[token_id])  # Fragments are still bytes, not strings.
    combined_bytes = b"".join(chunks)     # We put all the bytes together first.
    return combined_bytes.decode("utf-8")

for text in ["lowest", "hello world", "unseen 😀", " word", "word", '{"a":123}']:
    ids = bpe_encode(text, bpe_merges)
    print(repr(text), "code points:", len(text), "bytes:", len(text.encode("utf-8")))
    print("BPE IDs:", ids, "token count:", len(ids))
    assert bpe_decode(ids, bpe_pieces) == text

The text with emoji is also reconstructed, even if the emoji was not in the text of the rules creation; The base bytes are always in the dictionary. This feature is different from the limited character method of the previous example, which used zero for the unknown sign.

Pay attention to the distance between the first two samples `word`. Space is also an input to the tokenizer and can change identifiers. Number, markup and JSON text are also dependent on learned rules. The number of Persian tokens in this example is the result of this small training text; It does not rule about all token generators or all Persian models. In optional lesson 07, you will do this comparison with the actual `cl100k_base` token generator.

**Applicable exercise:** Make the number of integrations 8 and 32. Are the training text and the new text equally shorter? See the actual count; More rules do not guarantee equal compression of all inputs.

In [ ]:
comparison_texts = [training_texts[0], "hello world", "a completely unseen sentence"]
for max_merges in [8, 32]:
    pieces, merges = train_byte_bpe(training_texts, max_merges=max_merges)
    print("Requested merges:", max_merges, "learned merges:", len(merges))
    for text in comparison_texts:
        ids = bpe_encode(text, merges)
        print(repr(text), "tokens:", len(ids))
        assert bpe_decode(ids, pieces) == text

## 7. ID is a number; So why do we need vectors?

The ID is just the address. If "A" has an identifier of 1 and "B" has an identifier of 2, their one-unit difference does not describe a specific linguistic feature. Instead of using this number as a numeric value, the model takes its corresponding row from a table.

This table is called **Embedding**. Each line is a vector of decimal numbers. You saw a manual example in lesson 00; Here you check its connection with token generator and training. The numbers in this table are trainable weights, that is, the program can change them to reduce the target error.

First, create a manual table with four rows and two columns. Four rows for four identifiers and two columns for two numbers of each vector. The meaning of the columns in this example is only our chosen numbers; They do not measure "meaning" or "matter".

In [ ]:
manual_table = torch.tensor([
    [0.0, 0.0],   # Line ID zero
    [10.0, 11.0], # ID line one
    [20.0, 21.0], # ID line two
    [30.0, 31.0], # ID line three
])
manual_ids = torch.tensor([[2, 1, 2]], dtype=torch.long)  # We have a text with three tokens.
manual_vectors = manual_table[manual_ids]  # Each identifier is replaced by its own line.
print("Table shape:", tuple(manual_table.shape))
print("ID shape:", tuple(manual_ids.shape))
print("Vector shape:", tuple(manual_vectors.shape))
print("Vectors:", manual_vectors)
assert torch.equal(manual_vectors[0, 0], manual_vectors[0, 2])

The format of the identifiers is `(1, 3)` and the format of the output is `(1, 3, 2)`: one text, three positions and two numbers for each position. The first and third lines of the output are equal, because they both had an identifier of 2. This operation is called **Lookup**; It means removing the value from the table with its address.

In the common naming of models, `B` is the number of texts in a group or **Batch**, the letter `T` is the number of tokens in each text, the letter `C` is the number of numbers in each vector, and the letter `V` is the size of words. So the table has the form `(V, C)` and the output has the form `(B, T, C)`. These names are only abbreviations after seeing the real numbers.

### Make the same table with a teachable tool

The `nn` section of PyTorch has tools for building neural networks. The command `nn.Embedding(V, C)` creates a teachable table. Its initial values ​​are random. Input IDs must be integers; That's why you use `dtype=torch.long`.

In this example, the size of the vector is 8 to keep the output small. The size of the table is the actual word size of the project. Choice 8 is for the training of this section; The complete model in lesson 04 has its own settings.

In [ ]:
from torch import nn

torch.manual_seed(7)  # We fix the initial values ​​of this table in the execution again.
embedding = nn.Embedding(tokenizer.vocab_size, 8)
ids = torch.tensor([tokenizer.encode("model")], dtype=torch.long)
token_vectors = embedding(ids)  # The embedding lookup returns the selected rows.
print("Table shape:", tuple(embedding.weight.shape))
print("IDs:", ids)
print("ID shape:", tuple(ids.shape))
print("Vector shape:", tuple(token_vectors.shape))
print("First token vector:", token_vectors[0, 0].detach().tolist())

# We check that the ready-made tool has done exactly the same removal of table rows.
assert torch.equal(token_vectors, embedding.weight[ids])

In the output figure, the last dimension is 8. The name `embedding.weight` refers to the trainable table itself. The `detach` command displays a version of the result unrelated to the gradient calculation; It does not change the table values.

Basic random vectors are not reliable for semantic application. Even if they happen to have two vectors close together, no target has yet been taught that makes this closeness useful.

## 8. Which lines get gradients in education?

In lesson 00, **gradient** showed how the error changes with the change of a weight. The embedding table has the same weight. If only the lines corresponding to the "model" are used in a calculation, the direct gradient reaches the same lines in the same calculation.

To see this behavior, define a test error: the square of all the numbers of the selected vectors and then their average. This goal makes the numbers go to zero. **This goal is not to teach the language model**; It's just a small way to see the gradient flow. You examine the purpose of the language model in lessons 04 and 05.

The `backward` command stores the gradient in the `.grad` weight section. As in lesson 00, the weight itself does not change with `backward`; Updating is a separate step.

In [ ]:
embedding.zero_grad(set_to_none=True)  # We clear the gradient of the previous possible run.
token_vectors = embedding(ids)
objective = token_vectors.square().mean()  # square calculates the second power and mean calculates the average.
objective.backward()  # We calculate the sensitivity of this error to each weight in the table.

rows_with_grad = []
for row_id, row_gradient in enumerate(embedding.weight.grad):
    # If the sum of the absolute magnitudes is positive, then at least one of the row numbers has a gradient.
    if row_gradient.abs().sum().item() > 0:
        rows_with_grad.append(row_id)

used_rows = sorted(set(ids.flatten().tolist()))  # flatten puts all identifiers in one dimension.
print("Toy objective:", objective.item())
print("Rows used by the input:", used_rows)
print("Rows with direct gradient:", rows_with_grad)
assert rows_with_grad == used_rows

The two lists of rows must be equal. If a letter is repeated in the input, the same line is used several times and the effects are accumulated in the gradient of that line. The output of this example about the direct gradient is the same Lookup; In a complete model, other use of tables or update rules can create different behavior.

**Goal of the next example:** See also the update step. With a factor of 0.1, you subtract part of the gradient from the table. This is the manual rule of lesson 00. When changing the weight, you temporarily turn off the registration of the gradient calculation with `torch.no_grad()`.

In [ ]:
before_update = embedding.weight.detach().clone()  # clone makes an independent copy of the table.
with torch.no_grad():
    embedding.weight -= 0.1 * embedding.weight.grad  # It is a step towards reducing experimental error.

after_objective = embedding(ids).square().mean()
actually_changed = []
for row_id in range(tokenizer.vocab_size):
    if not torch.equal(before_update[row_id], embedding.weight[row_id]):
        actually_changed.append(row_id)
print("Objective before:", objective.item())
print("Objective after:", after_objective.item())
print("Changed rows:", actually_changed)
assert after_objective.item() < objective.item()
assert actually_changed == used_rows

The test error should be reduced and the changed lines should be the same lines used. This time the weight really changed, because you did the attribution. The final value of the table is not saved in this lesson and does not replace the weights of the original model.

### Is the vector of a token fixed or dependent on the sentence?

In the original embedding table, an identifier always takes the same row. But the model then processes this vector along with the other vectors of the sentence. The **Attention** operation, which you will see in the next lesson, allows the display of a position to be influenced by its previous text. So "Vector removed from the table" and "Token display after sentence processing" are two different steps.

## 9. How to measure the proximity of vectors?

To understand the similarity measure, choose three 2D hand vectors: `(1, 0)`, `(2, 0)`, and `(0, 1)`. The first and second vectors point in the same direction, but the second vector is twice as long. The third vector is perpendicular. This is a geometric example; These vectors are not derived from a text model.

**Internal multiplication or Dot product** means the multiplication of collocation numbers and their sum. **vector norm** in this section means the root of the sum of the squares of its numbers, not the number of members. Cosine similarity is the inner product divided by the product of the length of two vectors. For two non-zero vectors, this value is between negative one and one. A value of one means equal direction, zero means perpendicular direction and negative one means opposite direction.

**Objective of the example:** Perform the calculation without hiding the details of the formula. The symbol `** 0.5` calculates the square root.

In [ ]:
vector_a = torch.tensor([1.0, 0.0])
vector_b = torch.tensor([2.0, 0.0])
vector_c = torch.tensor([0.0, 1.0])

dot_ab = (vector_a * vector_b).sum()
length_a = vector_a.square().sum() ** 0.5
length_b = vector_b.square().sum() ** 0.5
cosine_ab = dot_ab / (length_a * length_b)
euclidean_ab = (vector_a - vector_b).square().sum() ** 0.5
print("Dot(a,b):", dot_ab.item())
print("Lengths:", length_a.item(), length_b.item())
print("Cosine(a,b):", cosine_ab.item())
print("Euclidean distance(a,b):", euclidean_ab.item())

The inner product is 2, the lengths are 1 and 2, and the cosine is 1. The direct distance between two points, which is called Euclidean distance, is equal to 1. Therefore, the cosine of one does not mean that all the numbers of two vectors are equal; It only shows the same direction.

**Vector normalization** In this section, it means dividing its numbers by the length of the vector, until the length of the result becomes one. This operation is different from text normalization in the Unicode section. For vectors of length one, the inner product is equal to cosine. The `F.normalize` tool does this for each row. The zero vector has no defined direction; In interpreting its cosine, you should consider this limitation.

In the following code, the `@` operator is matrix multiplication. The `.T` statement moves rows and columns. The result of multiplying the table in the shifted table is the comparison of each row with each other row.

In [ ]:
import torch.nn.functional as F

geometric_vectors = torch.stack([vector_a, vector_b, vector_c])  # We make three lines of three vectors.
unit_vectors = F.normalize(geometric_vectors, dim=-1)  # The last dimension is the columns of each row.
dot_matrix = geometric_vectors @ geometric_vectors.T
cosine_matrix = unit_vectors @ unit_vectors.T
distance_matrix = torch.cdist(geometric_vectors, geometric_vectors)  # It gives the distance of each pair.
print("Dot product matrix:\n", dot_matrix)
print("Cosine matrix:\n", cosine_matrix)
print("Euclidean distance matrix:\n", distance_matrix)

# For unit vectors, the distance squared is equal to two minus two times the cosine.
unit_distances = torch.cdist(unit_vectors, unit_vectors)
assert torch.allclose(unit_distances.square(), 2 - 2 * cosine_matrix, atol=1e-6)

In the cosine table, the comparison of the first vector with the second is equal to 1 and its comparison with the third is equal to zero. In the internal multiplication table, the length of the vectors also has an effect; The comparison of the second vector with itself is equal to 4.

The choice of **Metric** depends on the purpose of training the model. If a model is trained to compare cosines, changing the criterion arbitrarily can change the order of the search results. For a real model, you check the order of using the same model and its quality on your examples.

## 10. What is the difference between token embedding and search embedding?

Token embedding is taught in this course's model to help predict the next character. Each token has an initial vector and then its representation is changed in the sentence. By themselves, these vectors are not a good tool for document search.

In **RAG**, abbreviated `Retrieval-Augmented Generation`, relevant parts of the documents are first found and then given to the model along with questions to generate answers. Here we call the document finding section **Retrieval**. For that, a separate model usually creates a vector from the question and from each piece of document. A "chunk" is a limited and specific part of the text of the document.

For example, you want to ask the question "How many days of annual leave?" Close to the vacation clause in the regulation. A model trained to compare texts pursues this goal. The simple average of small GPT character vectors of this course does not replace such training.

To choose a recovery model, you check a few things with real data:

- **Language:** Does it give a suitable result on the Persian text and question?
- **Input length:** How many tokens does it accept and how does it cut long text?
- **Input format:** Does it require separate prefix for question and document? Some models have seen this format in training.
- **Comparison criteria:** Should the vectors be normalized and cosine, internal multiplication or distance be used?
- **Execution cost:** How much memory and time do the model size and vector dimension require?
- **Retrieval quality:** Are really relevant documents at the top of the results list?

**Index** is a structure for storing and searching a vector of documents. If you change the embedding model or its vector dimension, you must rebuild the document vector and rebuild the profile with the appropriate settings. Even if the dimensions of two models are the same, their coordinates are not necessarily consistent.

Arbitrary removal of the last columns of a vector does not guarantee quality preservation. Some models are trained with the **Matryoshka** method so that a certain number of initial numbers of their vectors are useful. This feature must exist in the same model and the result of the smaller dimension should still be measured on the real data.

### Great similarity does not prove the truth of a claimThe two sentences "This product is suitable for children" and "This product is not suitable for children" are about the same subject. The embedding model may consider them to be close, while they claim to be the opposite. The same problem may occur with different numbers or different versions of a document.

So don't interpret the close vector as a "correct answer". Retrieval is the step of finding relevant text; Examining the content of that text is another step. In optional lesson 07, you implement a real multilingual embedding model. The hands-on example of this lesson has not measured the quality of that model.

## 11. Check understanding by changing the code

These exercises use the concepts of this file. First guess the answer and then run the code. The goal is to find input and output connections, not to memorize function names.

**Exercise 1:** In `mini_tokenizer`, how many tokens do the text "Bob" and "Bob Bob" have? Is distance a token?

**Exercise 2:** Replace token ID "B" with ID "A". If you don't change the old embedding table, which rows will the model remove? Why shouldn't this change be done after training?

**Exercise 3:** In the embedding example, change the text "model" to "mem". Compare the number of input positions and the number of distinct lines used. Then return the original text and re-run the cells from the embedding.

**Exercise 4:** Instead of the vector `(2, 0)`, put the vector `(-2, 0)`. What is the inner product and its cosine with `(1, 0)`?

**Exercise 5:** Does the BPE tokenizer of this lesson need to have the new emoji in its character vocabulary? Which component makes its exact reconstruction possible?

<!-- fa-lint-ignore-next-line -->
<details dir="rtl"><summary>Open the answers after solving the exercises</summary>

**First answer:** "Bob" has three tokens and "Bob Bob" has seven tokens; The distance is also seen in the vocabulary of this small token maker. Character count is not word count.

**Second answer:** The new ID selects another row from the old table. If the mapping is changed, the meaning of the stored weights and the output lines of the model will be inconsistent. You keep the tokenizer with weights and settings.

**Third answer:** The "Mem" input has two positions but only one unique identifier. The same line is removed twice and the effect of both uses is added to its gradient.

**4th answer:** Inner product is negative 2 and cosine is negative 1. The vectors have opposite directions. The negative cosine in the geometrical example does not mean "falseness of a sentence".

**5th Answer:** All base bytes are present. So the text is just converted to bytes and can be left unmerged. Reconstruction is done by pasting byte chunks and then reading UTF-8.

</details>

## 12. For the interview, build the answer with your own example**Question: Is a token always a word?** No; It depends on the token generator. In this project, each known code point is a token. In BPE, fragments are made up of merging small units and may be only part of a word.

**Question: What is the difference between ID and embedding?** ID is the address of a vocabulary member; Embedding is the vector of teachable numbers related to that member. Closeness of numbers is not a measure of semantic similarity.

**Question: Why do we keep the tokenizer with the model?** The weights of the embedding table and the output rows are trained with a specific token-to-id mapping. Changing the mapping without correspondingly changing the model will spoil the input and output interpretation.

**Question: What is the difference between cosine and inner multiplication?** Inner multiplication is also sensitive to the length of the vectors. Cosine puts the length in the denominator and compares the direction. After normalizing two non-zero vectors with length one, these two values ​​are equal.

**Question: Why is this GPT's internal embedding not enough for RAG?** Its training goal is to predict the next character. The purpose of retrieval is to bring together related questions and text and separate unrelated results. The quality of that second goal should be measured with the appropriate model and evaluation data.

**Next step:** [Lesson 03: How does the previous text affect the display of a token?] (03_attention_and_transformers.ipynb) In that lesson, you start from the vectors of the same path and calculate the Attention operation step by step.